In [1]:
import pickle
import os

import pandas as pd
from tqdm.notebook import tqdm

from matplotlib import pyplot as plt
import seaborn as sns
import plotly.express as px
import pandas as pd
plt.rcParams["figure.dpi"] = 200
sns.set_palette("deep")
sns.set_context("paper")
sns.set_style("whitegrid")
from pyphylon.util import load_config

In [2]:
import gzip

In [3]:
mash_scrubbed_metadata = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/interim/2b_ncbi_mash_scrubbed_species_metadata.csv', index_col=0, dtype='object')


In [4]:
mash_scrubbed_metadata = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/interim/2b_ncbi_mash_scrubbed_species_metadata.csv', index_col=0, dtype='object')

display(
    mash_scrubbed_metadata.shape,
    mash_scrubbed_metadata.head()
)

(1317, 83)

,genome_id,genome_name,organism_name,taxon_id,genome_status,strain,culture_collection,type_strain,completion_date,bioproject_accession,...,assembly_stats_genome_coverage,assembly_stats_atgc_count,organism_infraspecific_names_strain,organism_infraspecific_names_isolate,assembly_stats_total_number_of_chromosomes,wgs_project_accession,annotation_count_gene_total,annotation_count_gene_protein_coding,annotation_count_gene_pseudogene,mash_cluster
0,GCA_048593205.1,Pseudomonas aeruginosa ps1,Pseudomonas aeruginosa,287,Complete,ps1,NaN,NaN,2025-03-17,PRJNA1226941,...,50.0,6037891,ps1,NaN,1.0,NaN,5601.0,5416.0,121.0,62
1,GCA_051016185.1,Pseudomonas aeruginosa 25181,Pseudomonas aeruginosa,287,Complete,25181,NaN,NaN,2025-06-21,PRJNA1180571,...,34.0,7421260,25181,NaN,2.0,NaN,7102.0,6922.0,95.0,2
2,GCF_000006765.1,Pseudomonas aeruginosa PAO1,Pseudomonas aeruginosa PAO1,208964,Complete,PAO1,NaN,NaN,2006-07-24,PRJNA331,...,NaN,6264404,PAO1,NaN,1.0,NaN,5697.0,5572.0,19.0,34
3,GCF_000014625.1,Pseudomonas aeruginosa UCBPP-PA14,Pseudomonas aeruginosa UCBPP-PA14,208963,Complete,UCBPP-PA14,NaN,NaN,2006-10-06,PRJNA386,...,NaN,6537637,UCBPP-PA14,NaN,1.0,NaN,6041.0,5891.0,70.0,2
4,GCF_000026645.1,Pseudomonas aeruginosa LESB58,Pseudomonas aeruginosa LESB58,557722,Complete,LESB58,NaN,NaN,2008-12-24,PRJEA31101,...,NaN,6601757,LESB58,NaN,1.0,NaN,6195.0,6059.0,52.0,94


# Run Bakta

``` 
./run_bakta_annotation.py --output /mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/ --force --input /mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/mash/mash_genomes/
```

In [5]:
# List of bakta-annotated faa files (needed for CD-HIT)
# bakta is a software that annotate bacterial genomes, MAGs and plasmids
# MAGs: metagenome-assembled genome, genetic material that's directly from environmental samples (collective material from the microbial communities)
# advantage of bakta, dbxref-rich, database cross reference
# sORF: small open reading frame, has a AUG and a stop codon, short sequences that have the potential to encode small peptides

BAKTA = os.path.join('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/')

bakta_faa_paths = [
    os.path.join(BAKTA, bakta_folder, bakta_folder+'.faa') 
    for bakta_folder in os.listdir(BAKTA)
]

bakta_faa_paths[:5]

['/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_051416385.1/GCF_051416385.1.faa',
 '/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_036237115.1/GCF_036237115.1.faa',
 '/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_001900225.1/GCF_001900225.1.faa',
 '/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_014217315.1/GCF_014217315.1.faa',
 '/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_041344955.1/GCF_041344955.1.faa']

In [7]:
# Sanity check
for path in tqdm(bakta_faa_paths):
    assert os.path.isfile(path)

  0%|          | 0/1317 [00:00<?, ?it/s]

In [8]:
# ensure that bakta paths are in our PG
real_paths = []
for f in bakta_faa_paths:
    # count += 1
    for i in mash_scrubbed_metadata['genome_id'].tolist():
        ################### adding in '+ '/'' here to account for genome IDs that are substrings of other genome IDs (ex: 287.2981 and 287.29817)
        if i + '/' in f:
            print(f)  
            real_paths.append(f)

/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_051416385.1/GCF_051416385.1.faa
/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_036237115.1/GCF_036237115.1.faa
/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_001900225.1/GCF_001900225.1.faa
/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_014217315.1/GCF_014217315.1.faa
/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_041344955.1/GCF_041344955.1.faa
/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_031212645.1/GCF_031212645.1.faa
/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_028404085.1/GCF_028404085.1.faa
/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_001618925.1/GCF_001618925.1.faa
/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_019915445.1/GCF_019915445.1.faa
/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/bakta/GCF_001515915.2/GCF_001515915.2.faa
/mnt/craig/pan_phylon/P_aerugi

# CD-HIT

running CD-HIT to determine core genome threshold for running panaroo

In [ ]:
#### before running next step, copied all .faa files from bakta to 'all_faa_bakta'
#### cp **/*.faa /mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/all_faa_bakta

#### remove all ...hypotheticals.faa
#### rm *.hypotheticals.faa

#### then combined all faa file content to PAeruginosa.faa 
#### expand ../all_faa_bakta/*.faa > PAeruginosa.faa

#### with the all faa file, run cdhit
####### do not create the PAeruginosa folder beforehand
####### may have to delete and recreate the cd-hit-results folder
#######cdhit -i /mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/all_faa_bakta/PAeruginosa.faa -o /mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/cd-hit-results/PAeruginosa -d 0 -n 5 -c 0.8 -M 0 -aL 0.8 -T 8


In [9]:
from pyphylon.pangenome import build_cds_pangenome

df_alleles, df_genes, header_to_allele = build_cds_pangenome(
    genome_faa_paths=real_paths,
    output_dir = '/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/cd-hit-results/',
    name="PAeruginosa",
    cdhit_args={'-n': 5, '-c':0.8, '-aL':0.8, '-T': 0, '-M': 0},
    fastasort_path=None,
    save_csv=False
)

Identifying non-redundant CDS sequences...
Headers without sequences: 0


0it [00:00, ?it/s]

0it [00:00, ?it/s]

MISSING: CJKDBL_30770
MISSING: EDGKOL_29840
MISSING: JAPKNI_17700
Loadings header-allele mappings...
Sorting alleles...
Sorting clusters...


  0%|          | 0/3441071 [00:00<?, ?it/s]

Genomes: 1317
Clusters: 55754
Alleles: 3441071
Updating genome 1 : GCA_048593205.1 	Alleles: 5541 	Clusters: 5525
Updating genome 2 : GCA_051016185.1 	Alleles: 6870 	Clusters: 6746
Updating genome 3 : GCF_000006765.1 	Alleles: 5677 	Clusters: 5649
Updating genome 4 : GCF_000014625.1 	Alleles: 5903 	Clusters: 5870
Updating genome 5 : GCF_000026645.1 	Alleles: 6032 	Clusters: 5989
Updating genome 6 : GCF_000168335.1 	Alleles: 5919 	Clusters: 5897
Updating genome 7 : GCF_000223945.1 	Alleles: 6190 	Clusters: 6148
Updating genome 8 : GCF_000223965.1 	Alleles: 6161 	Clusters: 6126
Updating genome 9 : GCF_000226155.1 	Alleles: 5759 	Clusters: 5734
Updating genome 10 : GCF_000271365.1 	Alleles: 5848 	Clusters: 5828
Updating genome 11 : GCF_000271985.2 	Alleles: 5640 	Clusters: 5620
Updating genome 12 : GCF_000284555.1 	Alleles: 6199 	Clusters: 6163
Updating genome 13 : GCF_000414035.1 	Alleles: 5795 	Clusters: 5767
Updating genome 14 : GCF_000496455.2 	Alleles: 6582 	Clusters: 6460
Updating g

In [10]:
df_genes.sum()

GCA_048593205.1    5525
GCA_051016185.1    6746
GCF_000006765.1    5649
GCF_000014625.1    5870
GCF_000026645.1    5989
                   ... 
GCF_905071885.1    6191
GCF_951691365.1    6632
GCF_951802375.2    6363
GCF_951805275.2    6360
GCF_976988945.1    5894
Length: 1317, dtype: Sparse[int64, nan]